# Smartphone Review Sentiment Analysis & Aspect-Level Breakdown

Run all cells below in Google Colab to inspect the dataset, train and benchmark the 3 classifiers (`Logistic Regression`, `Linear SVM`, `Multinomial Naive Bayes`), view evaluation charts, and test live review + aspect predictions directly inside Colab.

In [ ]:
import os

if not os.path.exists("train_day1.py"):
    if not os.path.exists("smartphone-reviews-sentiment"):
        !git clone https://github.com/MachavarapuAbhiram/smartphone-reviews-sentiment.git
    %cd smartphone-reviews-sentiment

!pip install -q streamlit

In [ ]:
!python check_csv.py

In [ ]:
!python train_day1.py

In [ ]:
import pandas as pd
from IPython.display import Image, display

print("=== Model Comparison Leaderboard ===")
display(pd.read_csv("outputs/day1_model_metrics.csv"))

print("\n=== Confusion Matrix ===")
display(Image("outputs/confusion_matrix.png"))

print("\n=== Sample Misclassifications (Top 10) ===")
display(pd.read_csv("outputs/sample_misclassifications.csv").head(10))

In [ ]:
import joblib
from train_day1 import clean_review_text
import app

pipeline = joblib.load("models/best_sentiment_pipeline.joblib")
reviews_df = pd.read_csv("data/smartphone_reviews.csv")
reviews_df["sentiment"] = reviews_df["rating"].apply(app.map_rating_to_sentiment)

print("=== Dataset Aspect Overview ===")
display(app.compute_aspect_overview(reviews_df))

sample_review = (
    "Daylight camera is sharp and display colors look vibrant, but battery "
    "drains fast on mobile data and barely lasts till evening. Performance "
    "heats up quickly during BGMI gaming and lags badly."
)
cleaned = clean_review_text(sample_review)
pred, scores_df, _ = app.get_prediction_scores(pipeline, cleaned)
top_feats = app.get_top_influential_features(pipeline, cleaned, pred, top_n=10)
aspect_df = pd.DataFrame(app.analyze_aspects(pipeline, sample_review))

print(f"\nLive Review: {sample_review}")
print(f"Predicted Overall Sentiment: {pred}")
display(scores_df)
print("\n=== Top Influential N-grams ===")
display(top_feats)
print("\n=== Aspect-Level Breakdown ===")
display(aspect_df)